In [1]:
import os, json, pickle
import numpy as np
import pandas as pd
import networkx as nx

ROOT    = "/teamspace/studios/this_studio/misinformation-detection"
OUTPUTS = f"{ROOT}/outputs"

# load bigcn results
bigcn_preds     = np.load(f'{OUTPUTS}/bigcn_preds.npy')
bigcn_probs     = np.load(f'{OUTPUTS}/bigcn_probs.npy')
bigcn_valid_idx = np.load(f'{OUTPUTS}/bigcn_valid_idx.npy')

# load texts and labels
with open(f'{OUTPUTS}/texts_and_labels.json') as f:
    tl = json.load(f)
texts  = tl['texts']
labels = np.array(tl['labels'])

# load cascade features
df = pd.read_csv(f'{OUTPUTS}/cascade_features.csv')

# load graphs
with open(f'{OUTPUTS}/all_cascades.pkl', 'rb') as f:
    cascade_data = pickle.load(f)

all_cascades = []
for c in cascade_data:
    G = nx.DiGraph()
    G.add_nodes_from(c['nodes'])
    G.add_edges_from(c['edges'])
    all_cascades.append((G, c['label'], c['root_id'], c['event']))

print(f"Loaded {len(bigcn_preds)} predictions over {len(all_cascades)} cascades")

Loaded 5447 predictions over 5802 cascades


In [2]:
# align true labels with valid_idx
true_labels = labels[bigcn_valid_idx]

# find misclassified
misclassified_mask = bigcn_preds != true_labels
misclassified_idx  = np.where(misclassified_mask)[0]

print(f"Total misclassified : {len(misclassified_idx)}")
print(f"Total evaluated     : {len(bigcn_preds)}")
print(f"Error rate          : {len(misclassified_idx)/len(bigcn_preds):.2%}")

# split into FP and FN
# FP: true=0 (non-rumour), predicted=1 (rumour)
# FN: true=1 (rumour),     predicted=0 (non-rumour)
fp_mask = (true_labels == 0) & (bigcn_preds == 1)
fn_mask = (true_labels == 1) & (bigcn_preds == 0)

fp_idx = np.where(fp_mask)[0]
fn_idx = np.where(fn_mask)[0]

print(f"\nFalse Positives (non-rumour → predicted rumour) : {len(fp_idx)}")
print(f"False Negatives (rumour → predicted non-rumour) : {len(fn_idx)}")

# confidence = probability assigned to the WRONG class
# for FP: model assigned high prob to rumour (class 1) but true is non-rumour
# for FN: model assigned high prob to non-rumour (1 - prob) but true is rumour
fp_confidence = bigcn_probs[fp_idx]           # high prob → model very confident it's rumour
fn_confidence = 1 - bigcn_probs[fn_idx]       # high value → model very confident it's non-rumour

# sort by confidence descending — most confidently wrong first
fp_sorted = fp_idx[np.argsort(fp_confidence)[::-1]]
fn_sorted = fn_idx[np.argsort(fn_confidence)[::-1]]

# take top 8 FP and top 7 FN (mix of both, 15 total)
top_fp = fp_sorted[:8]
top_fn = fn_sorted[:7]

print(f"\nTop 8 FP selected (highest confidence wrong)")
print(f"Top 7 FN selected (highest confidence wrong)")

Total misclassified : 929
Total evaluated     : 5447
Error rate          : 17.06%

False Positives (non-rumour → predicted rumour) : 453
False Negatives (rumour → predicted non-rumour) : 476

Top 8 FP selected (highest confidence wrong)
Top 7 FN selected (highest confidence wrong)


In [3]:
def get_cascade_info(local_idx):
    """Given index in valid_idx space, return full cascade info."""
    global_idx  = bigcn_valid_idx[local_idx]
    G, label, root_id, event = all_cascades[global_idx]
    
    text        = texts[global_idx]
    true_label  = true_labels[local_idx]
    pred_label  = bigcn_preds[local_idx]
    prob        = bigcn_probs[local_idx]
    confidence  = prob if pred_label == 1 else 1 - prob
    
    # cascade features
    feats = df.iloc[global_idx]
    
    # get reply texts (first 3)
    non_root = [n for n in G.nodes() if n != root_id]
    reply_texts = [G.nodes[n].get('text', '') for n in non_root[:3]]
    
    return {
        'global_idx'    : global_idx,
        'event'         : event,
        'true_label'    : 'rumour' if true_label == 1 else 'non-rumour',
        'pred_label'    : 'rumour' if pred_label == 1 else 'non-rumour',
        'error_type'    : 'FP' if (true_label==0 and pred_label==1) else 'FN',
        'confidence'    : round(float(confidence), 4),
        'source_text'   : text,
        'reply_texts'   : reply_texts,
        'cascade_size'  : feats['size'],
        'depth'         : feats['depth'],
        'branching'     : feats['branching'],
        'growth_30m'    : feats['growth_30m'],
        'burstiness'    : feats['burstiness'],
    }

# build full analysis table
records = []
for idx in top_fp:
    records.append(get_cascade_info(idx))
for idx in top_fn:
    records.append(get_cascade_info(idx))

error_df = pd.DataFrame(records)
print(f"Error analysis table: {error_df.shape}")
print(error_df[['event','true_label','pred_label','confidence',
                'cascade_size','depth']].to_string())

Error analysis table: (15, 13)
                event  true_label  pred_label  confidence  cascade_size  depth
0         sydneysiege  non-rumour      rumour         1.0            17      4
1         sydneysiege  non-rumour      rumour         1.0             8      1
2   germanwings-crash  non-rumour      rumour         1.0             3      1
3        charliehebdo  non-rumour      rumour         1.0            20      4
4      ottawashooting  non-rumour      rumour         1.0            51     19
5            ferguson  non-rumour      rumour         1.0             3      1
6   germanwings-crash  non-rumour      rumour         1.0             4      1
7        charliehebdo  non-rumour      rumour         1.0             6      2
8        charliehebdo      rumour  non-rumour         1.0             6      1
9   germanwings-crash      rumour  non-rumour         1.0             2      1
10           ferguson      rumour  non-rumour         1.0            39      5
11  germanwings-crash

In [4]:
for i, row in error_df.iterrows():
    print(f"\n{'='*70}")
    print(f"Example {i+1} | {row['error_type']} | Event: {row['event']}")
    print(f"True: {row['true_label']} | Predicted: {row['pred_label']} | "
          f"Confidence: {row['confidence']:.4f}")
    print(f"Cascade size: {row['cascade_size']} | Depth: {row['depth']} | "
          f"Branching: {row['branching']} | Growth 30m: {row['growth_30m']}")
    print(f"\nSource tweet:")
    print(f"  {row['source_text']}")
    print(f"\nFirst 3 replies:")
    for j, reply in enumerate(row['reply_texts']):
        print(f"  [{j+1}] {reply}")


Example 1 | FP | Event: sydneysiege
True: non-rumour | Predicted: rumour | Confidence: 1.0000
Cascade size: 17 | Depth: 4 | Branching: 9 | Growth 30m: 17

Source tweet:
  SYDNEY ATTACK
- Hostages at Sydney cafe
- 13 hostages
- 2 gunmen
- Hostages seen holding Arabic flag

Stay with @PzFeed for the latest.

First 3 replies:
  [1] Their leader Adnan Hoca and he uses the his cats to blow up bombs please careful sons of Jesus @babetsihayvan @pzfeed
  [2] They are from Turkey and they want speak Ottoman language in Turkey.. @PzFeed
  [3] @Normaldegildir Türkiyenin en güvenilir takipçi satın alma sitesi .www.takipcisatinal.com

Example 2 | FP | Event: sydneysiege
True: non-rumour | Predicted: rumour | Confidence: 1.0000
Cascade size: 8 | Depth: 1 | Branching: 7 | Growth 30m: 8

Source tweet:
  #SydneySiege
- 5 people injured
- Reports of at least 2 dead
- Bomb disposal unit inside cafe
- Police confirm siege is over.

First 3 replies:
  [1] @AusNewsNetwork @KennyTomlinson
  [2] RT @AusNewsN

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("=== Error Analysis Summary ===\n")

print("Error type breakdown:")
print(error_df['error_type'].value_counts())

print("\nEvent distribution in errors:")
print(error_df['event'].value_counts())

print("\nAverage cascade features — FP vs FN:")
print(error_df.groupby('error_type')[['cascade_size','depth',
                                       'branching','growth_30m']].mean())

# compare error cascade sizes vs overall
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# cascade size distribution
axes[0].hist(df['size'], bins=30, alpha=0.5, label='All cascades', color='#378ADD')
axes[0].hist(error_df['cascade_size'], bins=10, alpha=0.7,
             label='Misclassified', color='#E8772E')
axes[0].set_xlabel('Cascade size')
axes[0].set_ylabel('Count')
axes[0].set_title('Cascade size: errors vs all')
axes[0].legend()

# confidence distribution of errors
fp_confs = error_df[error_df['error_type']=='FP']['confidence']
fn_confs = error_df[error_df['error_type']=='FN']['confidence']
axes[1].hist(fp_confs.values, bins=8, alpha=0.7, label='FP', color='#E8772E')
axes[1].hist(fn_confs.values, bins=7, alpha=0.7, label='FN', color='#7F77DD')
axes[1].set_xlabel('Model confidence (wrong prediction)')
axes[1].set_ylabel('Count')
axes[1].set_title('Confidence of wrong predictions')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUTS}/error_analysis_plots.png', dpi=150)
plt.show()

# save error table
error_df.to_csv(f'{OUTPUTS}/error_analysis.csv', index=False)
print("\nSaved error_analysis.csv")

=== Error Analysis Summary ===

Error type breakdown:
error_type
FP    8
FN    7
Name: count, dtype: int64

Event distribution in errors:
event
sydneysiege          5
germanwings-crash    4
charliehebdo         3
ferguson             2
ottawashooting       1
Name: count, dtype: int64

Average cascade features — FP vs FN:
            cascade_size     depth  branching  growth_30m
error_type                                               
FN             16.285714  2.857143  10.714286   11.714286
FP             14.000000  4.125000   6.000000   10.000000
